# Разведка API

API: `http://final-project.simulative.ru/data/`

Документации нет, поэтому изучаю опытным путём: какой метод, какие параметры, есть ли авторизация, что возвращает за пустой день, с какой даты вообще есть данные.

In [ ]:
import requests
import json
import pandas as pd
from pathlib import Path

URL = "http://final-project.simulative.ru/data/"

# папка для сырых ответов
Path("raw").mkdir(exist_ok=True)

## Первый запрос

Известно, что дата передаётся в формате `yyyy-mm-dd`. Попробую самый простой вариант — GET с датой в query.

In [ ]:
r = requests.get(URL, params={"date": "2024-06-01"})
print("Статус:", r.status_code)
print("Первые 300 символов ответа:")
print(r.text[:300])

Статус 200 — значит запрос ушёл и сервер ответил. В ответе видны записи о покупках. Авторизация не нужна.

## Структура ответа

In [ ]:
data = r.json()

print("Тип:", type(data).__name__)
print("Записей за день:", len(data))
print("\nПример одной записи:")
data[0]

Ответ — список словарей. Одна запись = одна позиция в чеке (то есть если в одной покупке несколько товаров, они придут разными записями).

Поля:
- `client_id` — id покупателя
- `gender` — пол
- `purchase_datetime` — дата покупки
- `purchase_time_as_seconds_from_midnight` — время в секундах от полуночи
- `product_id` — id товара
- `quantity` — количество
- `price_per_item` — цена за штуку (без скидки)
- `discount_per_item` — скидка за штуку
- `total_price` — итоговая сумма по позиции

Проверю, что цены сходятся: `(price_per_item - discount_per_item) * quantity` должно равняться `total_price`.

In [ ]:
row = data[0]
calc = (row["price_per_item"] - row["discount_per_item"]) * row["quantity"]
print("Расчёт:", calc)
print("В ответе:", row["total_price"])
print("Совпадает:", calc == row["total_price"])

Сходится. Значит `total_price` — это уже итог с учётом скидки и количества, отдельно умножать не нужно.

## Удобно посмотреть в виде таблицы

In [ ]:
df = pd.DataFrame(data)
df.head()

In [ ]:
df.dtypes

## Пагинация

7000+ записей — нестандартное число (не 1000 и не 5000), значит лимита по страницам, скорее всего, нет. Проверю — посмотрю заголовки ответа, обычно если есть пагинация, в них упоминаются `total`, `page`, `link`.

In [ ]:
for k, v in r.headers.items():
    print(f"{k}: {v}")

В заголовках про пагинацию ничего нет. Значит API отдаёт за один запрос все записи за день — это удобно, не нужно собирать страницы.

## Что приходит за пустой день

Проверю на дате, для которой данных точно не должно быть — например, 2010 год.

In [ ]:
r_empty = requests.get(URL, params={"date": "2010-01-01"})
print("Статус:", r_empty.status_code)
print("Ответ:", r_empty.text)

Статус 200, но в ответе обычная строка `"Информация за более ранние периоды отсутствует"` — это **не JSON**. Значит признак «данных за этот день нет» — ответ не парсится как JSON.

Это важно учесть в скрипте загрузки: оборачивать `r.json()` в `try/except`, и если падает — пропускать день.

In [ ]:
# Проверю, как именно ловить пустой ответ
try:
    r_empty.json()
    print("Распарсилось — данные есть")
except ValueError:
    print("Не парсится — данных нет")

## С какой даты начинаются данные

Раз на 2010г данных нет, а на 2024г есть надо найти границу. Буду тыкать разные годы и сужать диапазон.

In [ ]:
# 2018 — есть данные?
r_test = requests.get(URL, params={"date": "2018-01-01"})
print("2018-01-01:", r_test.status_code, "—", r_test.text[:80])

In [ ]:
# 2020?
r_test = requests.get(URL, params={"date": "2020-01-01"})
print("2020-01-01:", r_test.status_code, "—", r_test.text[:80])

In [ ]:
# 2022?
r_test = requests.get(URL, params={"date": "2022-01-01"})
print("2022-01-01:", r_test.status_code, "—", r_test.text[:80])

С 2022-01-01 данные уже есть, а с 2020-01-01 ещё нет. Значит граница где-то в 2021 году. Проверю 2021-й по середине.

In [ ]:
r_test = requests.get(URL, params={"date": "2021-06-01"})
print("2021-06-01:", r_test.status_code, "—", r_test.text[:80])

In [ ]:
r_test = requests.get(URL, params={"date": "2021-12-31"})
print("2021-12-31:", r_test.status_code, "—", r_test.text[:80])

Весь 2021 год — пусто, а с 2022-01-01 — данные. **Самая ранняя дата с данными: `2022-01-01`.**

Значит истории больше 4 лет, должно быть достаточно.

## Сохраню сырой ответ

In [ ]:
# полный ответ за день
with open("raw/2024-06-01_full.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

# маленький пример из 3 записей — чтобы потом удобно показать структуру
with open("raw/2024-06-01_sample.json", "w", encoding="utf-8") as f:
    json.dump(data[:3], f, ensure_ascii=False, indent=2)

print("Сохранено в папку raw/")

## Итоги разведки

| Параметр              | Значение |
|-----------------------|----------|
| Метод                 | GET |
| Параметр даты         | `date` в query |
| Формат даты           | `yyyy-mm-dd` |
| Авторизация           | не нужна |
| Тип ответа            | JSON-массив |
| Пагинация             | нет, день целиком |
| Записей в день        | ~7000 |
| Самая ранняя дата     | 2022-01-01 |
| Признак пустого дня   | ответ не парсится как JSON |

Дальше буду проектировать схему БД под эти данные.